First we look at the current integral controller


In [67]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np

tau_hb = 10 # samplingsfrekvens, once per 10 seconds
v_n = 20826690 /(1e9) # measured ppb drift
print("v_n: ", v_n)
# v_n = 0.00005
z = ct.tf('z')
G_temp = (1+v_n )*tau_hb / (z - 1) # ct.tf cannot take dt when getting 'z'

z = ct.tf('z')
G = ct.tf(G_temp.num, G_temp.den, dt=tau_hb) 
print(G)
plt.figure(1)
ct.pzmap(G)
plt.title("Pole/zero plot for plant")
plt.figure(2)
ct.bode_plot(G, display_margins=True)
plt.title("Bode plot of plant")
plt.show()


v_n:  0.02082669
<TransferFunction>: sys[1143]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = 10
  10.21
  -----
  z - 1


In [59]:
hr = np.linspace(0, 50, 10000)
Gi = 1/( 1 - z**(-1))
ct.rlocus(G*Gi , gains=hr)
plt.title("Possible conjugate pairs of poles using I-controller")
plt.show()


Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


In [68]:
Kp = 0.04
Ki = 0.06

Gpi = (Kp*(z-1) + Ki*z)/(z-1)
print(Gpi)
print()
Gcl = ct.feedback(G*Gpi, 1)
print(Gcl)
poles = ct.poles(Gcl)
plt.figure(1)
ct.pzmap(Gcl)
plt.title("Pole/zero plot of PI-controller")
print(poles)
plt.figure(2)
ct.bode_plot(G*Gpi, display_margins=True)
plt.title("Bode plot of closed loop")
plt.show()


<TransferFunction>: sys[1155]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
  0.1 z - 0.04
  ------------
     z - 1
<TransferFunction>: sys[1158]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = 10
     1.021 z - 0.4083
  -----------------------
  z^2 - 0.9792 z + 0.5917
[0.48958666+0.59327416j 0.48958666-0.59327416j]


In [69]:
# Assuming tau_hb, z, and G are already defined in your previous cell

# 1. Define sets of (Kp, Ki) to experiment with
pi_settings = [
    (.04, .06),   # Your original values
    (0.001, 0.0005),  # Slower, less aggressive
    (0.0008, 0.0002)    # Higher proportional gain
]

# 2. Create the time vector and ramp input
# Let's simulate for 60 samples (600 seconds)
num_samples = 10
T = np.arange(0, num_samples * tau_hb, tau_hb)
U_ramp = T # A standard unit ramp where input equals time

plt.figure(figsize=(10, 6))

# 3. Loop through settings and simulate
for Kp, Ki in pi_settings:
    # Define Controller
    Gpi = (Kp*(z-1) + Ki*z) / (z-1)
    
    # Define Closed Loop
    Gcl = ct.feedback(G * Gpi, 1)
    
    # Simulate Ramp Response
    time, response = ct.forced_response(Gcl, T, U_ramp)
    
    # Plot this specific response
    plt.step(time, response, where='post', label=f'Kp={Kp}, Ki={Ki}')

# Plot the ideal reference ramp (what we want the system to track)
plt.plot(T, U_ramp, 'k--', linewidth=2, label='Ideal Reference Ramp')

plt.title("Closed-Loop Ramp Response with Different PI Tunings")
plt.xlabel("Time (seconds)")
plt.ylabel("System Output")
plt.legend()
plt.grid(True)
plt.show()
